# Continual learning for second-life EV battery health: the complete project in one notebook

21CSC314P Big Data Essentials, Review 2. Runs on Google Colab (CPU is enough; about 25 minutes). It uses the repository's own code and its committed results, so every number shown traces to a results file written on a clean commit (`docs/RESULTS.md`, `docs/DECISIONS.md`).

| Section | What it shows | Source |
|---|---|---|
| 1. Setup | clone, pinned libraries, Java for Spark | repository |
| 2. Tests | the unit tests (synthetic data in temporary folders only) | `tests/` |
| 3. Big-data preprocessing | full-dataset preprocessing results and the Spark validation; then a **live run** of the Spark pipeline on a checksummed real subset of S6 | committed results; live demo |
| 4. Forecasting benchmarks | S1 and S6 continual stream (P1), unseen cells or conditions (P2) with reuse classes, first-life pretraining (P3) | committed results |
| 5. Proposed model vs existing models | residual TFT + EWC against the neural variants and the conventional baselines: RMSE, paired tests with the replication verdict, forgetting, per-group errors, figures | committed results (D23) |
| 6. RC-TFT and literature baselines (D24) | shown when their results are committed | committed results, if present |
| 7. Conclusions | what the results support, and the limitations | — |

**Definitions.** SOH = capacity / first valid measured capacity × 100 (relative SOH, not nominal capacity). The task is to forecast SOH 1–3 (E1) or 1–6 (E2) capacity checks ahead. Errors are RMSE in percentage points (pp) of SOH. Continual-learning runs: 10 task orders × 5 seeds per seed set (main: seeds 0–4, replicate: seeds 5–9). A difference counts as a finding only if it replicates (D21 rule: Holm-corrected p < 0.05 in both seed sets, same sign, |difference| ≥ 0.05 pp in both).

## 1. Setup

The repository is private, so cloning needs a GitHub token with read access to it:

1. On GitHub: Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token. Repository access: only this repository; permissions: Contents = Read-only. Copy the token.
2. In Colab: the key icon (Secrets) in the left sidebar → Add new secret, name `GITHUB_TOKEN`, value = the token, and switch on notebook access.

Without the secret, the next cell asks for the token in a hidden prompt instead. The token is used for the clone only and is removed from the clone's configuration afterwards; it never appears in the notebook's output.

The project's libraries are installed into a separate environment (`/content/venv`), so the session never needs a restart: one **Runtime → Run all** runs the whole notebook (about 25 minutes).

In [ ]:
BRANCH = "main"
REPO = "TarunSitaraman/continual-learning-heterogeneous-second-life-ev-batteries"

import os, subprocess
from getpass import getpass
if not os.path.isdir("/content/repo"):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:   # secret missing or notebook access not granted
        token = getpass("GitHub token with read access to the repository (input hidden): ")
    clone = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, f"https://{token}@github.com/{REPO}.git",
                            "/content/repo"], capture_output=True, text=True)
    del token
    if clone.returncode != 0:
        raise SystemExit("Clone failed: check that the token has read access to " + REPO)
    # Remove the token from the clone's saved remote.
    subprocess.run(["git", "-C", "/content/repo", "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
os.chdir("/content/repo")
print(subprocess.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short"], capture_output=True, text=True).stdout)

In [ ]:
%%capture install_log
# The project's pinned libraries (requirements.txt) go into a separate environment, /content/venv, which inherits
# Colab's own packages (torch). Tests, Spark and every script run there; this notebook's Python keeps Colab's
# libraries and only reads files and draws tables and figures, so no session restart is needed.
import os, shutil, subprocess, sys
os.chdir("/content/repo")
VENV = "/content/venv"
PY = VENV + "/bin/python"
if not os.path.exists(PY):
    !{sys.executable} -m pip install -q uv
    !{sys.executable} -m uv venv -q --system-site-packages --python {sys.executable} {VENV}
    !grep -v -E "^(jupyter|torch)==" /content/repo/requirements.txt > /tmp/requirements_colab.txt
    !{sys.executable} -m uv pip install -q --python {PY} -r /tmp/requirements_colab.txt
if subprocess.run([PY, "-c", "import torch"]).returncode != 0:      # outside Colab: no inherited torch
    !{sys.executable} -m uv pip install -q --python {PY} torch --index-url https://download.pytorch.org/whl/cpu
if not shutil.which("java"):
    !apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless

In [ ]:
import glob, os, subprocess, sys
os.chdir("/content/repo")   # also after a session restart
java = (sorted(glob.glob("/usr/lib/jvm/java-17-openjdk-*")) or sorted(glob.glob("/usr/lib/jvm/*")))[0]
os.environ["JAVA_HOME"] = java
os.environ["PATH"] = java + "/bin:" + os.environ["PATH"]
PY = "/content/venv/bin/python"   # project environment (pinned libraries)
check = subprocess.run([PY, "-c", "import sys, numpy, pandas, sklearn, torch, pyspark; print('project environment: Python', "
                        "sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| scikit-learn', "
                        "sklearn.__version__, '| torch', torch.__version__, '| PySpark', pyspark.__version__)"],
                       capture_output=True, text=True)
print(check.stdout or check.stderr)
assert check.returncode == 0, "the project environment is incomplete: see install_log.show() in the previous cell"
!java -version 2>&1 | head -1
import importlib.util, json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
REPO_ROOT = Path("/content/repo")
spec = importlib.util.spec_from_file_location("d24_outputs", REPO_ROOT / "scripts" / "make_d24_outputs.py")
O = importlib.util.module_from_spec(spec)
spec.loader.exec_module(O)   # table and verdict code shared with the paper tables; its main() is not called
%matplotlib inline

def show(fig_name, width=760):
    p = REPO_ROOT / "paper" / "figures" / fig_name
    if p.exists():
        display(Image(filename=str(p), width=width))
    else:
        print("missing figure:", fig_name)

## 2. Tests

In [ ]:
!{PY} -m pytest -q -p no:cacheprovider 2>&1 | tail -3

## 3. Big-data preprocessing

### 3.1 Results on the full data (committed)

The S6 between-check ageing data: 827 files, about 132 GB raw, 901,974,259 rows (counted). Apache Spark (PySpark, local mode on one machine) maps each file to per-cycle responses, caches the per-cycle table and reduces it with Spark SQL to 21 inputs per (cell, 100-cycle block). The Spark output is checked against an independent pandas implementation on every block.

In [ ]:
!{PY} scripts/make_preprocessing_table.py > /dev/null
display(pd.read_csv("results/tables/preprocessing.csv")[["dataset", "stage", "value", "unit"]])
v = json.loads(Path("results/d22_spark_validation.json").read_text())
print({k: v[k] for k in ("engine", "blocks", "cycles", "seconds", "compared_blocks", "criterion", "matches_pandas")})
print("largest |Spark - pandas| over all inputs:", max(d["max_abs"] for d in v["per_input"].values()))
a = json.loads(Path("results/d22_ageing_report.json").read_text())
print("causality check:", a["causality"]["blocks_checked"], "blocks,", a["causality"]["violations_at_valid_checks"],
      "violations at valid checks")
show("spark_pipeline.png", 900)

### 3.2 Live run on a real subset of S6

Downloads (checked against the checksums committed in `docs/external_data/metadata/`): the authors' `extractedData.mat` (MD5), the capacity-check files of cell 1a (CRC-32, read from its zip with HTTP range requests) and one eighth of cell 1a's ageing files (about 4 files of roughly 1.6 million rows each). Labels are rebuilt and checked against the authors' capacities; the ageing inputs are extracted with pandas and with Spark, and compared. Numbers in this subsection come from the subset: a demonstration of the pipeline, not reportable results. Manifests of this partial download go to `/content`, so the committed manifests stay unchanged.

In [ ]:
RUN_REAL_DATA_DEMO = os.environ.get("RUN_REAL_DATA_DEMO", "1") == "1"   # set to False to skip the downloads
CELL = "Cell1a"

import hashlib, json, urllib.request
from pathlib import Path
raw = Path("data/raw/wheeler2025"); raw.mkdir(parents=True, exist_ok=True)
record = json.loads(Path("docs/external_data/metadata/wheeler2025_lfp_dataverse.dataset.json").read_text())
files = record["data"]["latestVersion"]["files"] if "data" in record else record["latestVersion"]["files"]
mat = next(f["dataFile"] for f in files if f["dataFile"]["filename"] == "extractedData.mat")
target = raw / "extractedData.mat"
if RUN_REAL_DATA_DEMO and not target.exists():
    urllib.request.urlretrieve(f"https://entrepot.recherche.data.gouv.fr/api/access/datafile/{mat['id']}", target)
if RUN_REAL_DATA_DEMO:
    md5 = hashlib.md5(target.read_bytes()).hexdigest()
    assert md5 == mat["md5"], f"MD5 mismatch: {md5} != {mat['md5']}"
    print("extractedData.mat MD5 verified:", md5)
else:
    print("live demo switched off: section 3.2 is skipped")

In [ ]:
if RUN_REAL_DATA_DEMO:
    !{PY} scripts/fetch_wheeler_characterisation.py --cells {CELL} --manifest /content/charac_manifest.json
    !{PY} scripts/build_wheeler_features.py --workers 2

In [ ]:
if RUN_REAL_DATA_DEMO:
    checks = pd.read_csv("data/wheeler_checks.csv")
    print(f"{len(checks)} checks of {checks['cell'].nunique()} cell(s); {int(checks['valid'].sum())} with a C/25 label")
    display(checks[["cell", "Battery_ID", "cycle", "check_time", "Capacity_Ah", "SOH_Percent", "valid"]].head(10))

In [ ]:
if RUN_REAL_DATA_DEMO:
    !{PY} scripts/fetch_wheeler_characterisation.py --cells {CELL} --kind aging --shard 0/8 --manifest /content/aging_manifest.json

**Pandas reference** (`scripts/build_s6_ageing_features.py`): per-cycle responses (end-of-charge and end-of-discharge voltage, CV hold time and final current, relaxation after charge and discharge) and block aggregates (mean, last value, least-squares slope).

In [ ]:
if RUN_REAL_DATA_DEMO:
    !{PY} scripts/build_s6_ageing_features.py --workers 2

**Apache Spark** (`scripts/spark_ageing_features.py`): one map task per ageing file parses it into per-cycle responses; the per-cycle table is cached; Spark SQL groups by (cell, block) into the same aggregates; the output is compared with the pandas reference: |Spark − pandas| ≤ 1e-9·|pandas| + 1e-12 and the same missing values.

In [ ]:
if RUN_REAL_DATA_DEMO:
    !{PY} scripts/spark_ageing_features.py --cores 2 --tag colab
    v = json.loads(Path("results/d22_spark_validation_colab.json").read_text())
    print({k: v[k] for k in ("engine", "blocks", "cycles", "seconds", "compared_blocks", "matches_pandas")})
    print("largest |Spark - pandas|:", max(d["max_abs"] for d in v["per_input"].values()))

In [ ]:
if RUN_REAL_DATA_DEMO:
    blocks = pd.read_csv("data/s6_ageing_blocks_spark_colab.csv")
    display(blocks.iloc[:, :8])

The demo rewrote `results/d22_ageing_report.json` and other files for its subset; every committed results file is restored from Git before the results sections.

In [ ]:
!git checkout -- results paper
!git status --short results paper | head -5; echo "results restored"

## 4. Forecasting benchmarks (committed results)

### 4.1 Continual stream (P1): S1 (six second-life NMC cells) and S6 (seven LFP usage conditions)
Final average test RMSE over the stream (pp), skill = 1 − RMSE / RMSE of persistence, forgetting in pp. Inputs of D18 (S1) and D20 (S6).

In [ ]:
p1 = pd.read_csv("results/tables/p1.csv")
for ds in ("S1", "S6"):
    print(ds)
    display(p1[p1["dataset"] == ds][["name", "rmse_pp_mean", "rmse_pp_sd", "n_runs", "skill_vs_persistence",
                                      "forgetting_pp_mean"]].sort_values("rmse_pp_mean").round(3))
show("12_p1_rmse.png", 640); show("13_p1_rmse.png", 640)

### 4.2 Unseen cell or usage condition (P2) and reuse classes
Leave one group out; S6 reuse classes C0–C3 from the forecast SOH (accuracy and macro-F1 against the measured class).

In [ ]:
p2 = pd.read_csv("results/tables/p2.csv")
display(p2.sort_values(["dataset", "rmse_pp_mean"]).round(3))

### 4.3 First-life pretraining (P3) and the input ablation
Pretraining on 369 first-life cells (S7–S9, 36,711 samples) before the S6 stream; final average test RMSE on S6, main / replicate seeds.

In [ ]:
p3 = json.loads(Path("results/forecasting_d21_p3.json").read_text())
s = pd.DataFrame(p3["summary"]).pivot(index="method", columns="set", values="final_avg_error_mean").round(3)
display(s.sort_values("main"))
print("baselines:", {b["method"]: round(b["rmse_final_mean_over_tasks"], 3) for b in p3["baselines"]})
show("d21_p3_final_error.png", 640); show("d21_ablation_a1.png", 640)

## 5. Proposed model against existing models (D23, committed results)

Residual TFT + EWC (forecast = linear extrapolation + a correction learned by a Temporal Fusion Transformer with Elastic Weight Consolidation) against the direct TFT, the MLP variants and the conventional models (linear extrapolation, gradient boosting, ridge, persistence), on the S6 stream: E1 = horizons 1–3, E2 = horizons 1–6. Tests: paired two-sided Wilcoxon against the reference, Holm-corrected within each seed set; `diff` = reference − other (negative favours residual TFT + EWC).

In [ ]:
REF = "tft_residual_ewc"
data = {}
for exp in ("e1", "e2"):
    path = REPO_ROOT / "results" / f"forecasting_d23_{exp}.json"
    data[exp] = json.loads(path.read_text())
    env = data[exp]["environment"]
    print(f"{exp}: {path.name}, decision {data[exp]['decision']}, profile {data[exp]['profile']}, "
          f"horizons {data[exp]['horizons']}, {data[exp]['n_samples']} samples, commit {env['git_commit'][:7]}, "
          f"git_dirty={env['git_dirty']}")

### 5.1 Final test RMSE of every method

In [ ]:
def comparison_view(exp):
    c = O.comparison(data[exp])
    view = c[["label", "kind", "rmse_main_mean", "rmse_main_sd", "rmse_replicate_mean", "rmse_replicate_sd",
              "forgetting_main"]].rename(columns={
        "label": "method", "rmse_main_mean": "RMSE main (pp)", "rmse_main_sd": "s.d. main",
        "rmse_replicate_mean": "RMSE replicate (pp)", "rmse_replicate_sd": "s.d. replicate",
        "forgetting_main": "forgetting main (pp)"})
    return c, view

comp = {}
for exp in ("e1", "e2"):
    comp[exp], view = comparison_view(exp)
    print(f"{exp.upper()} (horizons {data[exp]['horizons']}), sorted by main RMSE")
    display(view.round(3))

### 5.2 Paired tests against residual TFT + EWC, with the replication verdict

In [ ]:
tests = {}
for exp in ("e1", "e2"):
    t = O.tests_table(data[exp])
    tests[exp] = t
    print(f"{exp.upper()}: reference = {t['reference'].iloc[0]}")
    display(t[["metric", "other", "diff_main", "p_holm_main", "diff_replicate", "p_holm_replicate", "verdict"]]
            .round({"diff_main": 3, "diff_replicate": 3}).style.format({"p_holm_main": "{:.2g}", "p_holm_replicate": "{:.2g}"}))

#### Summary: residual TFT + EWC against the conventional (non-neural) models

In [ ]:
conventional = [O.label(m) for m in ("linear_extrapolation", "gbm_shared", "ridge_schedule", "ridge_shared", "persistence")]
rows = []
for exp in ("e1", "e2"):
    t = tests[exp]
    t = t[(t["metric"] == "final_avg_error") & t["other"].isin(conventional)]
    rows += [{"experiment": exp.upper(), "baseline": r.other, "diff main (pp)": r.diff_main,
              "diff replicate (pp)": r.diff_replicate, "verdict": r.verdict} for r in t.itertuples()]
display(pd.DataFrame(rows).round(3))

### 5.3 Figure: final test RMSE (main seed set, mean ± s.d. over 50 runs)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharey=False)
for ax, exp in zip(axes, ("e1", "e2")):
    c = comp[exp].sort_values("rmse_main_mean", ascending=False)
    colours = ["#2a78d6" if m == REF else O.FAMILY_COLOUR.get(O.family(m), O.GREY) if O.family(m) else O.GREY
               for m in c["method"]]
    alpha = [1.0 if m == REF else 0.55 for m in c["method"]]
    bars = ax.barh(c["label"], c["rmse_main_mean"], xerr=c["rmse_main_sd"].fillna(0), color=colours,
                   height=0.65, error_kw={"elinewidth": 0.8, "ecolor": O.INK2})
    for b, a in zip(bars, alpha):
        b.set_alpha(a)
    lin = c.loc[c["method"] == "linear_extrapolation", "rmse_main_mean"].iloc[0]
    ax.axvline(lin, color=O.INK, linestyle="--", linewidth=0.8)
    ax.set_xlabel("final average test RMSE (pp of SOH)")
    ax.set_title(f"{exp.upper()}: horizons {data[exp]['horizons'][0]}–{data[exp]['horizons'][-1]} "
                 f"(dashed: linear extrapolation)")
    for lbl in ax.get_yticklabels():
        if lbl.get_text() == O.label(REF):
            lbl.set_fontweight("bold")
fig.suptitle("S6 continual stream: residual TFT + EWC (bold, blue) vs neural variants and conventional baselines (grey)")
fig.tight_layout()
plt.show()

### 5.4 Forgetting: what EWC adds to the residual TFT

In [ ]:
rows = []
for exp in ("e1", "e2"):
    runs = pd.DataFrame(data[exp]["runs"])
    for st in ("main", "replicate"):
        r = runs[runs["set"] == st]
        for m in ("tft_residual_ewc", "tft_residual_finetune"):
            g = r[r["method"] == m]
            rows.append({"experiment": exp.upper(), "set": st, "method": O.label(m),
                         "RMSE (pp)": g["final_avg_error"].mean(), "forgetting (pp)": g["forgetting"].mean()})
display(pd.DataFrame(rows).round(3))
t = pd.concat([tests[e].assign(experiment=e.upper()) for e in ("e1", "e2")])
display(t[t["other"] == O.label("tft_residual_finetune")][["experiment", "metric", "diff_main", "p_holm_main",
                                                           "diff_replicate", "p_holm_replicate", "verdict"]].round(4))

### 5.5 Per test case (per-group metrics; main seeds, test RMSE after the last stream step)

In [ ]:
for exp in ("e1", "e2"):
    print(exp.upper())
    display(O.per_group(data[exp]).round(3))

### 5.6 Error by horizon (E2) and per test case (figures)

In [ ]:
show("d24_horizon_e2.png", 560); show("d24_per_group_e1.png", 760)

## 6. RC-TFT and the literature baselines (D24)

**RC-TFT** (Residual Continual TFT) is the proposed extension: the residual TFT above trained with *shift-adaptive* EWC, which protects an earlier battery group more when a new group's inputs differ more from it. It is compared with literature models: BiLSTM, TCN and a Transformer encoder (continual, fine-tuning and EWC), and GPR, random forest and SVR (fitted once on all training rows). The protocol was fixed before any run (D24 in `docs/DECISIONS.md`). This cell shows whatever D24 results are committed.

In [ ]:
base_files = sorted(Path("results").glob("forecasting_d24_baselines_e*.json"))
full_files = sorted(Path("results").glob("forecasting_d24_e[12].json"))
if full_files:
    !{PY} scripts/summarise_d24.py
elif base_files:
    for f in base_files:
        b = json.loads(f.read_text())
        print(f"{f.name}: commit {b['environment']['git_commit'][:7]}, git_dirty={b['environment']['git_dirty']}")
        display(pd.DataFrame(b["baselines"])[["method", "final_avg_error"]].assign(label=lambda t: t["method"].map(O.label))
                .sort_values("final_avg_error").round(3))
    print("RC-TFT and the BiLSTM/TCN/Transformer baselines: not committed yet.")
else:
    print("No D24 results are committed yet: RC-TFT and the literature baselines have not been run on S6.")

## 7. Conclusions

### Proposed model against existing models (section 5)

Every number below is printed by the cells above (diff = TFT + EWC − other, main / replicate, pp; verdicts by the D21 rule).

**Against the conventional models, residual TFT + EWC is better, and every such difference replicates, except against linear extrapolation:**
- Gradient boosting: −0.112 / −0.138 (E1) and −0.337 / −0.365 (E2).
- Ridge (schedule and shared inputs) and persistence: between −0.74 and −2.13 in both experiments.
- Linear extrapolation: practically equivalent in E1 (−0.004 / −0.029). Not replicated in E2 on the final error (+0.076 / +0.049), where linear extrapolation is lower; it is replicated better at horizons 1 and 2 (+0.189 / +0.144 and +0.103 / +0.076), and the two are practically equivalent at horizons 5 and 6.

**Against the other neural continual learners:**
- Residual TFT + EWC beats the direct TFT and direct MLP variants on final error, with every difference replicated except against direct fine-tuning (MLP in E1, TFT in both).
- Against the residual MLP it is replicated better than MLP fine-tuning in E1 only. Against MLP + EWC the E1 gap is below 0.05 pp in the main set, and in E2 the main set is not significant.

**What EWC adds** (residual TFT + EWC vs residual TFT fine-tuning):
- E1: practically equivalent on every metric.
- E2: forgetting is lower, and this replicates (0.093 vs 0.282 pp in the main set, 0.086 vs 0.220 in the replicate); backward transfer also replicates. The lower final error in E2 is not replicated.

**Claim for the paper:** a TFT that learns a correction to linear extrapolation, with EWC, is the best of the learned models and clearly outperforms gradient boosting, ridge and persistence, but it does not beat simple linear extrapolation on S6. EWC's replicated benefit is reduced forgetting over the longer horizons (E2). These are results on S6 only, with SOH relative to the first RPT capacity, so they are not comparable with literature numbers obtained on other datasets or SOH definitions.

### Big data, benchmarks and limitations
- **Preprocessing:** 901,974,259 raw rows reduced by Spark to 827 block rows of 21 inputs; Spark equals pandas on every block, and no block ends after a valid capacity check (section 3.1). The live run in section 3.2 repeats the pipeline on a real subset.
- **Benchmarks:** forgetting is negligible across the near-identical S1 cells and substantial across the S6 usage conditions; on unseen S6 conditions the learned models beat linear extrapolation and classify reuse classes C0–C3 far better than persistence (section 4.2); first-life pretraining does not improve S6 forecasts (section 4.3).
- **Limitations:** small numbers of groups (6 S1 cells; 2–3 S6 cells per condition); SOH relative to the first measured capacity, so results are not comparable with literature numbers on other datasets or SOH definitions; Spark ran on one machine in local mode, not on a cluster; S6 "second life" is continued ageing at 50 °C under the same protocol.

### Data sources and licences
- S6: W. Wheeler, Y. Bultel, P. Venet and A. Sari, "An ageing study of twenty 18650 lithium-ion Graphite/LFP cells in first and second life use," *Scientific Data* 12, 392 (2025); dataset doi:10.57745/OLBXKT, CC BY 4.0.
- S1: K. Moy *et al.*, *Data in Brief* 57, 111046 (2024); dataset doi:10.17605/OSF.IO/8JNR5 (version 2), CC BY 4.0.
- S7–S9 (pretraining pool): see `docs/EXTERNAL_DATA.md`, which records the citation, licence and hashes of every source.